# Run and evaluate a sales agent API

A sales manager wants to ask which region led in March and compare that month's sales with February. In this lab, you call a running API, follow its tool calls, and check the results.

Before running the cells, follow the [setup in the README](README.md#setup). Leave the server terminal open so you can see which tools run. Both Ollama and Groq use the same requests. If your server uses a port other than 8000, set `API_URL` in the first code cell to that address.

| Section | What you'll do |
|---|---|
| 1 | Check that the API is up and send a sales question |
| 2 | See how a route chooses an agent and a model chooses a tool |
| 3 | Check the answer, missing month, and blank input |
| 4 | Assemble context for a follow-up request |

## 1. Call the API

FastAPI connects HTTP routes to Python functions. Uvicorn runs the app and listens on a port. Your notebook is a client that sends requests to that port.

Try `GET /health` in [the interactive API docs](http://127.0.0.1:8000/docs), or run the cell below. Expect 200 and `{"status": "ok"}`. This confirms that the app is up. The March question in the next cell checks whether the app can run the agent.

In [ ]:
import os

import httpx

API_URL = os.getenv("SALES_API_URL", "http://127.0.0.1:8000").rstrip("/")

health = httpx.get(f"{API_URL}/health", timeout=5)
health.raise_for_status()
print(health.status_code, health.json())

Each POST sends a JSON object with a `question` field. A successful response has an `answer` field. The server lets the agent run for up to 30 seconds before requesting cancellation. The client has a 40-second timeout to give the server time to send its error response.

In the sample report, North has sales of $700 and $500. South has $900 and East has $600. Work out the winner before sending the request.

In [ ]:
def ask_api(route: str, question: str) -> httpx.Response:
    # httpx serializes this dictionary as the JSON request body.
    return httpx.post(
        f"{API_URL}{route}", json={"question": question}, timeout=40
    )


march = ask_api("/chat", "Which region had the highest sales in March?")
march.raise_for_status()
march_answer = march.json()["answer"]
print(march.status_code, march_answer)

Expect North and $1,200. The wording may vary. In the Uvicorn terminal, look for `sales_by_region` and the elapsed time.

If the server replies with a 4xx or 5xx status, `raise_for_status()` raises an `HTTPStatusError`. Read the status and the error response before looking for `answer`. If the server is unreachable or the client times out, the HTTP call itself raises a request error.

## 2. Follow the agent and its tools

| Route | Agent | Available tools |
|---|---|---|
| `POST /chat` | Sales agent | `sales_by_region` |
| `POST /report` | Report agent | `list_report_months` and `compare_months` |

The route chooses the agent. The model can request a tool from that agent's list. The application runs the Python function and gives its result back to the model. The model uses the result to write the reply.

In [ ]:
available = ask_api("/report", "Which months are available in the sales report?")
available.raise_for_status()
print(available.json()["answer"])

comparison = ask_api("/report", "Compare total sales in February and March.")
comparison.raise_for_status()
comparison_answer = comparison.json()["answer"]
print(comparison_answer)

Expect February and March in the first answer. In the second, expect February at $1,800 and March at $2,700 across all regions. Find both tool names in the server log.

### Read the server code

Open [sales_agent_api.py](sales_agent_api.py) and follow these functions in order.

| Code | What to look for |
|---|---|
| `totals_by_region` | Python adds the matching rows. The model receives the result from the tool. |
| `build_model` | The `MODEL_PROVIDER` environment variable chooses Ollama or Groq at startup. |
| chat and report | Pydantic validates the request shape. The route checks what it needs and hands the question to an agent. |
| `run_agent` | `ainvoke()` gives the route something it can `await`. While it waits for the model over the network, the server can handle another request. |
| `wait_for` and `perf_counter` | `wait_for` adds a time budget. Subtracting two `perf_counter` readings measures elapsed seconds for the log. |

The agent also has a graph step limit. That covers repeated model and tool steps, while the timeout covers elapsed time. A quick calculation such as adding the sales values is a regular Python call.

To try Groq, use the server commands in the [API guide](04_sales_agent_api.md) and point `API_URL` at that server. The Groq key belongs in the server terminal's environment. The notebook only needs the API URL.

## 3. Check the response

A 200 response tells you the request succeeded. Check the answer separately to see whether it contains the right facts. In a production eval, also compare tool arguments and structured results. A word match can pass even when a sentence uses the word incorrectly.

Try three cases. The March question should return North and 1,200. A question with no month should get a clarification. Spaces alone should get 422.

In [ ]:
import re

# Normalize the comma before checking the amount.
answer_text = march_answer.lower().replace(",", "")
assert march.status_code == 200
assert "north" in re.findall(r"[a-z]+", answer_text)
assert "1200" in re.findall(r"\d+", answer_text)
print("March answer passed")

missing_month = ask_api("/chat", "Which region led?")
assert missing_month.status_code == 200
assert missing_month.json()["answer"] == "Which month should I check?"
print("Missing month passed")

blank = ask_api("/chat", "   ")
assert blank.status_code == 422
assert blank.json()["detail"] == "Please enter a question."
print("Blank input passed")

Open `chat()` in the script. The route lowercases the question and extracts letter groups with the pattern `[a-z]+`. It turns those groups into a set. `isdisjoint()` is true when that set shares no full month name with the month set. The route then returns the clarification before it calls the agent. For “Which region led in March?”, `re.findall()` returns `['which', 'region', 'led', 'in', 'march']`. That list contains `march`, so the check lets the request continue. If you say “Which region led?”, the route asks for a month. The check uses full names, so `Mar` also gets a clarification.

In this demo, `422` means the input failed validation and `504` means the agent reached its wait budget. These are standard HTTP status codes. The [API guide](04_sales_agent_api.md) explains the app's error mappings.

## 4. Build context for a follow-up

Send just “Write my Monday briefing.” This API starts each request with a fresh message list. It will ask for a month because it only sees this request.

In [ ]:
follow_up = ask_api("/chat", "Write my Monday briefing.")
follow_up.raise_for_status()
print(follow_up.json()["answer"])

To support a briefing, the application could store the report months per conversation and a confirmed format preference per user. It can then load those fields and fetch the data it needs. For this exercise, use two small dictionaries and reuse the replies from the earlier cells.

In [ ]:
from pprint import pprint

conversations = {
    ("priya", "monday-briefing"): {"month": "March", "compare_with": "February"}
}
preferences = {"priya": {"format": "Two short bullets"}}

briefing_context = {
    "task": "Write a Monday sales briefing.",
    "period": conversations[("priya", "monday-briefing")],
    "format": preferences["priya"]["format"],
    "sales_facts": [march_answer, comparison_answer],
}
# Keep the task first in the printed view.
pprint(briefing_context, sort_dicts=False)

The output shows a task, report months for this conversation, a preference, and the sales evidence. The dictionaries are a notebook example of what an application could load. The running API still treats each request independently.

**Context engineering** is deciding what goes into the next model call. For a long chat, a common approach is to keep recent turns in full, summarize older turns, and search saved history when an exact detail is needed. The app can store the full transcript and decide how much to send on each model call. **The harness** is the application code that runs and supervises the agent. Here it validates the input, chooses the agent, sets a timeout, and logs tool names.

Try changing the format for one request. A new dictionary can take the same fields and override just `format`, leaving the saved preference available for later briefings.

In [ ]:
one_time_context = {**briefing_context, "format": "One short paragraph"}
print("This request", one_time_context["format"])
print("Saved preference", preferences["priya"]["format"])

## Try it yourself

Send a question about April and check that the reply explains that the report has no data for April. Then design a test for an order lookup. If the user asks about order 123, check that the tool looked up 123 and that the response reflects the tool result.

Continue with [agent evals](01_agent_evals_and_context.ipynb) for a closer look at tool traces, or [context and memory](05_context_memory_and_agent_harness.ipynb) for saved preferences and a generated briefing.